# Cryptography (CC4017) -- Week 5

## Chalenge 1

Use OpenSSL to calculate the SHA256 value of the pdf slides of this week’s class. Check if it equals:
d51b15eeed16158b0a2d0d50c92e3b34f62140b7627b88dca62d4a27e8f0f569

openssl dgst -sha256 .../Cripto/week05/cryptoSlidesW5.pdf

### 1.1 What does this tell you about the integrity of the file?

We can conclude that the ingetrity of the file has not been compromised. Since the given hash result was the same as the the one obtained by us. This is because even a small change, like of one bit, of the binary of the file creates a completely different hash. Has such, since the hash was the one expected, we can say the file hasn't been tampered with.

### 1.2 Suppose you alter the first 4 bytes of the original pdf file, and recompute the SHA256 value of this altered file. How many bytes do you expect to be affected by this change?

Whatever the hash function is, a simple small change should create a complete different hash result, a random oracle, otherwise it would be easier to guess what the correspondence between passwords and hashes. As such, its to be expected a change in all or almost all bytes of the hash.

## Chalenge 2

Use python to crack the security of predictable passwords in crack_hash.py

In [8]:
from cryptography.hazmat.primitives import hashes
from binascii import hexlify, unhexlify
import os
import numpy as np

# The most common passwords of 2019.
passwds = ['123456','123456789','qwerty','password','1234567','12345678','12345','iloveyou','111111','123123','abc123','qwerty123','1q2w3e4r','admin','qwertyuiop','654321','555555','lovely','7777777','welcome']

### Non-salt version

# Get their hex versions
hex_passwds = []
for pwd in passwds:
	hex_passwds.append(hexlify(pwd.encode()))

# Hash all the passwords
hlist = []
for pwd in hex_passwds:
	digest = hashes.Hash(hashes.SHA256())
	digest.update(pwd)
	hlist.append(hexlify(digest.finalize()))




#### Salt version

# Random salt of 1 byte
salt_passwds = []
salt = os.urandom(1)

# The same passwords, but now with the random salt prepended
for pwd in hex_passwds:
	salt_passwds.append(salt+pwd)

# Hash all salted passwords
shlist = []
for pwd in salt_passwds:
	digest = hashes.Hash(hashes.SHA256())
	digest.update(pwd)
	shlist.append(hexlify(digest.finalize()))

### Lets mix it up
# numpy 1.5.0 required!
mixed_hlist = np.random.permutation(hlist)
mixed_shlist = np.random.permutation(shlist)

### Exercise 1 - Crack unsalted hashes
# You can use mixed_hlist, hex_passwds and hlist
# Produce a list cracked_pwds that has the list of hex passwords in the correct sequence

cracked_pwds = [None] * len(passwds)

for i in hex_passwds:
    digest = hashes.Hash(hashes.SHA256())
    digest.update(i)
    curr_hash = (hexlify(digest.finalize()))
    try:
        right_idx = np.where(mixed_hlist == curr_hash)[0][0] 
        cracked_pwds[right_idx] = i
    except IndexError:
        print(f"Hash '{curr_hash.decode('utf-8')}' not found in mixed_hlist.")
 
 
# Lets see if your list is correct
i = 0
for pwd in cracked_pwds:
	digest = hashes.Hash(hashes.SHA256())
	digest.update(pwd)
	if (mixed_hlist[i] == hexlify(digest.finalize())):
		print(i, "Check")
	i += 1

### Exercise 2 - Crack salted hashes
# You can use mixed_shlist, hex_passwds and hlist
# You can't use shlist and salt!!
# Produce a list cracked_spwds that has the list of hex passwords in the correct sequence

cracked_spwds = [None] * len(passwds)

all_salts = [bytes([i]) for i in range(256)]

for pwd in hex_passwds:
    for s in all_salts:
        curr_spass = s + pwd
        digest = hashes.Hash(hashes.SHA256())
        digest.update(curr_spass)
        curr_hash = (hexlify(digest.finalize()))
        try:
            right_idx = np.where(mixed_shlist == curr_hash)[0][0] 
            cracked_spwds[right_idx] = s+pwd
        except IndexError:
            None

i = 0
for pwd in cracked_spwds:
	digest = hashes.Hash(hashes.SHA256())
	digest.update(pwd)
	if (mixed_shlist[i] == hexlify(digest.finalize())):
		print(i, "Check")
	i += 1


0 Check
1 Check
2 Check
3 Check
4 Check
5 Check
6 Check
7 Check
8 Check
9 Check
10 Check
11 Check
12 Check
13 Check
14 Check
15 Check
16 Check
17 Check
18 Check
19 Check
0 Check
1 Check
2 Check
3 Check
4 Check
5 Check
6 Check
7 Check
8 Check
9 Check
10 Check
11 Check
12 Check
13 Check
14 Check
15 Check
16 Check
17 Check
18 Check
19 Check


The attack on unsalted hashes is faster than the salted hashes. This happens because the speed of the first attack is O(n x m), where n is the number of passwords and m the number of hashes, which is the same, so O(n²) while the attack on salted hashes is O(2⁸×n²). The 2⁸ comes from the 8 bits of entropy added from the addition of randomly created salts, forcing an attacker to go over the 256 different possibibilites for each and every password. This means it takes 2⁸ times longer to find the correspondence to the 20 passwords and hashes with salt.

## Chalenge 3


Use the tool available here (or any other tool that works) to construct two PDFs with the same SHA-1 value.
Check out the SHAttered paper and explain how the attack works.

The SHAttered attack against SHA-1 is a collision attack, where two different inputs ,in this case, two different PDF files, are crafted to produce the same hash value. The goal is, then, to produce a collision while hashing two different files. With brute force, that should happen after 2⁸⁰ tries. In this attack, the goal was to approach the theoretical limit, around only 2⁶⁰ tries.

For the attack to be done, there is a series of pre-requisites that need to be true. Finding a collision with randomly generated files is very hard, as such, they had to create two files that share a lot of common structure but differ in carefully chosen parts. Both files started from a common prefix, meaning these files were carefully designed to contain the same headers, body structures, and formatting metadata. After that, we need to find near collisions, two inputs that are almost identical in hash value but differ in a few specific bits. 
This is done based on looking for neutral bits, which are used to make local adjustments while refining candidate message pairs for the near-collision, letting the attackers try many combinations without messing up the differential path, and boomerang bits are used during the early rounds of SHA-1’s compression function, introducing differences that temporarily propagate but will eventually cancel themselves out, allowing larger changes while still guiding the hash function toward a collision. With all these calculations they try to generate many collision candidates, which they iterated over and pruned, in order to avoid repeating solutions. With all this information collected, they carefully tweaked the two files, by making specific adjustments in the second half of the file, making it so the files were still valid, but different enough to be distinct. After all these challenges, all that is left is for the attacker to hash the files, which will end up producing the same hash. 

So, starting from a common prefix, we craft controlled differences in the blocks, propragate and search all possible near collisions that may be helpful, finalize the collision using the best sollution, and hash the files, creating a collision

## Chalenge 4
